# Training pipeline skeleton, Phase 1

I build the skeleton the training roles use: I load the clean training tables, I check
the audio folders match, I define the shared config, and I show what the
skeleton gives the team.

Contents:
1. Load the clean training tables
2. Verify the audio folders hold the same counts
3. The shared config
4. Look at a sample of the training table
5. Output format for results
6. What this skeleton gives every role

In [1]:
from pathlib import Path

import pandas as pd


def find_repo_root(start):
    p = Path(start).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError(
        "Repo root not found. Run this notebook from inside the "
        "smart-voicemail-router repo."
    )


REPO_ROOT = find_repo_root(Path.cwd())

# Data lives in this repo on the ready branch (dataset/ and final/ at the
# repo root). On a role branch the data comes from the svr-data worktree
# next to the repo, so fall back to that layout when the in-repo folders
# are not present.
if (REPO_ROOT / "final").is_dir() and (REPO_ROOT / "dataset").is_dir():
    DATA = REPO_ROOT
else:
    DATA = REPO_ROOT.parent / "svr-data"
TABLES = DATA / "final"
AUDIO = DATA / "dataset"

# Counts are for dataset v2. Update them when the dataset version changes
# (Phase 7 retrain on audited labels).
shared_config = {
    "split_seed": 42,
    "experiment_seeds": [42, 1337, 2023, 7, 99],
    "max_seq_len": 128,
    "split_ratio": "70/20/10",
    "dataset_version": "v2",
    "dataset": "svr-voicemail-corpus",
    "train_rows": 3994,
    "val_rows": 1130,
    "test_rows": 576,
}

import json

# Save the config so other notebooks load it instead of copying it by hand.
CONFIG_PATH = REPO_ROOT / "notebooks" / "training" / "shared_config.json"
CONFIG_PATH.write_text(json.dumps(shared_config, indent=2))
print(f"Shared config saved to {CONFIG_PATH.name}")


Shared config saved to shared_config.json


## 1. Load the clean training tables

I load the three tables from the smart-voicemail-router worktree. The tables come from the
SVR Voicemail Corpus: 5,700 clean rows after preparation (51 worst transcripts excluded, logprob below -0.50). The labels in these tables come from the model.

In [2]:
tables = {}
REQUIRED_COLS = ["audio_file", "text_norm", "complaint_category",
                 "complaint_urgency", "customer_state", "churn_risk"]
VALID_LABELS = {
    "complaint_category": ["network_and_connectivity", "billing_and_charges",
        "balance_and_data", "app_and_account", "branch_and_staff",
        "no_reply_or_long_wait", "cancellation_request", "thanks_or_praise",
        "other", "offers_and_packages", "sim_services"],
    "complaint_urgency": ["high", "medium", "low"],
    "customer_state": ["calm", "frustrated", "mad", "satisfied", "unclear"],
    "churn_risk": ["at_risk", "not_at_risk"],
}
for split in ["train", "val", "test"]:
    path = TABLES / f"clean_training_{split}.csv"
    tables[split] = pd.read_csv(path)
    n = len(tables[split])
    expected = shared_config[f"{split}_rows"]
    print(f"{split}: {n} rows from {path.name}")
    assert n == expected, f"{split} has {n} rows, expected {expected}"
    missing = [col for col in REQUIRED_COLS if col not in tables[split].columns]
    assert not missing, f"{split} is missing columns: {missing}"
    for col in REQUIRED_COLS:
        n_null = int(tables[split][col].isna().sum())
        assert n_null == 0, f"{split} has {n_null} nulls in {col}"
    n_empty = int((tables[split]["text_norm"].astype(str).str.strip() == "").sum())
    assert n_empty == 0, f"{split} has {n_empty} empty text_norm rows"
    for col, valid in VALID_LABELS.items():
        bad = sorted(set(tables[split][col].dropna().unique()) - set(valid))
        assert not bad, f"{split} has invalid {col} values: {bad}"

total = sum(len(t) for t in tables.values())
expected_total = (shared_config["train_rows"] + shared_config["val_rows"]
                  + shared_config["test_rows"])
print(f"total clean rows: {total}")
assert total == expected_total


train: 3994 rows from clean_training_train.csv
val: 1130 rows from clean_training_val.csv
test: 576 rows from clean_training_test.csv
total clean rows: 5700


## 2. Verify the audio folders hold the same counts

I count the WAV files in each split folder and I match each one by name
to its table row. The splits must match the audio folders before anyone trains.

In [3]:
for split in ["train", "val", "test"]:
    folder = AUDIO / split
    wav_names = {p.name for p in folder.iterdir() if p.suffix.lower() == ".wav"}
    table_files = set(tables[split]["audio_file"].tolist())
    n_wav = len(wav_names)
    n_rows = len(tables[split])
    print(f"{split}: {n_wav} WAV files, {n_rows} table rows")
    assert n_wav == n_rows, f"{split}: {n_wav} WAV files but {n_rows} rows"
    assert wav_names == table_files, (
        f"{split}: audio files and table rows do not match by name"
    )

print("Audio folders match the tables on every split, by count and by name.")


train: 3994 WAV files, 3994 table rows
val: 1130 WAV files, 1130 table rows
test: 576 WAV files, 576 table rows
Audio folders match the tables on every split, by count and by name.


## 3. The shared config

The shared config lives in the setup cell at the top. The loading cell reads its expected counts from it, and it is saved to shared_config.json so other notebooks load it instead of copying it by hand. One config, one seed setup, one split, so no one builds on a different setup. The config is printed again here so it is easy to see.


In [4]:
print(shared_config)


{'split_seed': 42, 'experiment_seeds': [42, 1337, 2023, 7, 99], 'max_seq_len': 128, 'split_ratio': '70/20/10', 'dataset_version': 'v2', 'dataset': 'svr-voicemail-corpus', 'train_rows': 3994, 'val_rows': 1130, 'test_rows': 576}


## 4. Look at a sample of the training table

I look at the first 3 rows of the training table to confirm the four label
fields are there. I print each row with its text and its four labels, so
the sample reads clearly on any screen: complaint_category,
complaint_urgency, customer_state, and churn_risk.

In [5]:
for i, row in tables["train"].head(3).iterrows():
    print(f"Row {i}: {row["text_norm"][:60]}...")
    print(f"  category: {row["complaint_category"]}")
    print(f"  urgency: {row["complaint_urgency"]}")
    print(f"  state: {row["customer_state"]}")
    print(f"  churn: {row["churn_risk"]}")
    print()


Row 0: لو سمحتوا ممكن حد يرجع معايا فاتوره الشغله ده المبلغ عالي من...
  category: billing_and_charges
  urgency: medium
  state: calm
  churn: not_at_risk

Row 1: في باقه مكتوب انها 30 يوم بس كل مره تتجدد بعد 28 يوم ده غش و...
  category: balance_and_data
  urgency: medium
  state: mad
  churn: not_at_risk

Row 2: سيد الخير انا حدثك بقالي اسبوع دلوقتي النت اقعد في البيت وحد...
  category: network_and_connectivity
  urgency: high
  state: frustrated
  churn: not_at_risk



## 5. Output format for results

I define one shape for results so the validation numbers from every role land in the same format. Every result record carries the run id, the model, the split, the seed, and the per-field macro-F1.


In [6]:
def make_result_record(run_id, model, split, seed):
    return {
        "run_id": run_id,
        "model": model,
        "split": split,
        "seed": seed,
        "metrics": {
            "complaint_category_macro_f1": None,
            "complaint_urgency_macro_f1": None,
            "customer_state_macro_f1": None,
            "churn_risk_macro_f1": None,
        },
    }


result_record = make_result_record("001", "EgyBERT", "val", shared_config["split_seed"])
print(result_record)


{'run_id': '001', 'model': 'EgyBERT', 'split': 'val', 'seed': 42, 'metrics': {'complaint_category_macro_f1': None, 'complaint_urgency_macro_f1': None, 'customer_state_macro_f1': None, 'churn_risk_macro_f1': None}}


## 6. What this skeleton gives every role

This skeleton gives every role four things:

- Data loading: one way to load the clean tables, with the counts, columns, nulls, empty text, and label values all checked.
- Shared config: one seed setup, one split, one dataset version, saved to shared_config.json for every training role to load.
- Seed handling: split_seed 42 for the split, and the same 5 experiment_seeds
  for every experiment, so results are comparable.
- Output format: one shape for results, so the validation numbers from every
  role land in the same format.

No one builds on a different setup. That is the point of Phase 1.